In [ ]:
from __future__ import annotations

import gc #Pembersih sampah pada ram
import time #Pengukur waktu eksekusi
from pathlib import Path #Manajemen alamat file/folder
import numpy as np #Operasi numerik
import pandas as pd #Manipulasi tabel data

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    BASE_DIR = Path('/content/drive/MyDrive/skripsi/data')
except ImportError:
    IN_COLAB = False
    BASE_DIR = Path('./data')

CLINVAR_DIR  = BASE_DIR / 'ClinVar'
DBNSFP_DIR   = BASE_DIR / 'dbNSFP'
CLEAN_CLINVAR      = CLINVAR_DIR / 'clinvar_cleaned_grch38.csv'
RAW_DBNSFP         = DBNSFP_DIR  / 'dbNSFP5.3a_grch38.gz'
FINAL_MERGED       = BASE_DIR / 'clinvar_dbnsfp_merged.csv'
CHECKPOINT_MERGED  = BASE_DIR / 'clinvar_dbnsfp_merged_checkpoint.csv'
CHECKPOINT_META    = BASE_DIR / 'clinvar_dbnsfp_merged_checkpoint.txt'

LAMPIRAN_DIR = BASE_DIR / 'Lampiran_Skripsi'
LAMPIRAN_DIR.mkdir(parents=True, exist_ok=True)

print('=' * 60)
print('TAHAP 3: ANOTASI dbNSFP')
print('=' * 60)

Mounted at /content/drive
TAHAP 3: ANOTASI dbNSFP


## HELPER: Tabel box rapi dengan garis Unicode

In [ ]:
# Mencetak dataframe pada tabel
def print_box_table(preview_df: pd.DataFrame) -> None:
    """Cetak DataFrame sebagai tabel box rapi dengan garis Unicode."""
    if preview_df.empty:
        print('   (tidak ada data untuk ditampilkan)')
        return

    display_df = preview_df.copy().fillna('')
    display_df = display_df.astype(str).map(
        lambda text: str(text).replace('\n', ' ').replace('\r', ' ')
    )
    headers = [str(c).replace('\n', ' ').replace('\r', ' ') for c in display_df.columns]
    rows = display_df.values.tolist()

    widths = []
    for i, header in enumerate(headers):
        cells = [row[i] for row in rows]
        widths.append(max(len(header), max((len(c) for c in cells), default=0), 3))

    def fmt_row(values: list) -> str:
        parts = [f' {str(v).ljust(widths[i])} ' for i, v in enumerate(values)]
        return '│' + '│'.join(parts) + '│'

    def border(left: str, mid: str, right: str) -> str:
        return left + mid.join('─' * (w + 2) for w in widths) + right

    print(border('┌', '┬', '┐'))
    print(fmt_row(headers))
    print(border('├', '┼', '┤'))
    for row in rows:
        print(fmt_row(row))
    print(border('└', '┴', '┘'))


def print_kv_table(data: dict, key_label: str = 'Item', val_label: str = 'Nilai') -> None:
    """Cetak dictionary sebagai tabel 2-kolom bergaris Unicode."""
    kw = max(len(key_label), max((len(str(k)) for k in data.keys()), default=0))
    vw = max(len(val_label), max((len(str(v)) for v in data.values()), default=0))

    def border(left: str, mid: str, right: str, fill: str) -> str:
        return left + fill * (kw + 2) + mid + fill * (vw + 2) + right

    print(border('┌', '┬', '┐', '─'))
    print(f'│ {key_label.ljust(kw)} │ {val_label.ljust(vw)} │')
    print(border('├', '┼', '┤', '─'))
    for k, v in data.items():
        print(f'│ {str(k).ljust(kw)} │ {str(v).ljust(vw)} │')
    print(border('└', '┴', '┘', '─'))

## HELPER: Parsing nilai semicolon dari dbNSFP

In [ ]:
# Mengambil angka tertinggi dari skornya
def extract_max_score(value: object) -> float:
    if pd.isna(value) or str(value).strip() in {'.', '', 'nan'}:
        return np.nan
    scores = []
    for item in str(value).split(';'):
        item = item.strip()
        # Lewati nilai kosong atau placeholder titik (.)
        if item and item != '.':
            try:
                scores.append(float(item))
            except ValueError:
                pass
    return max(scores) if scores else np.nan

## 1. CEK OUTPUT LAMA / CHECKPOINT

In [ ]:
SKIP_PIPELINE = False

# Memeriksa apakah file merged nya sudah ada
if FINAL_MERGED.exists():
    print('\nCheckpoint ditemukan: hasil gabungan sudah tersedia.')
    preview = pd.read_csv(FINAL_MERGED, nrows=20)
    n_total = sum(1 for _ in open(FINAL_MERGED)) - 1
    print_kv_table({
        'File'         : str(FINAL_MERGED),
        'Total Baris'  : f'{n_total:,}',
        'Total Kolom'  : str(preview.shape[1]),
    }, key_label='Info', val_label='Nilai')
    print('\n20 Data Teratas (Cached):')
    print_box_table(preview.head(20))

    top20_path = LAMPIRAN_DIR / 'clinvar_dbnsfp_merged_top20.csv'
    preview.head(20).to_csv(top20_path, index=False)
    print(f'Top 20 data tersimpan di: {top20_path}')

# Jika sudah ada maka akan ditampilkan
    if RAW_DBNSFP.exists():
        dbnsfp_preview_path = LAMPIRAN_DIR / 'dbnsfp_raw_preview_top20.csv'
        if not dbnsfp_preview_path.exists():
            print('\n[LAMPIRAN] Membuat preview isi raw dbNSFP (20 baris, kolom relevan)...')
            _df_hdr = pd.read_csv(RAW_DBNSFP, sep='\t', compression='gzip', nrows=0)
            _all_cols = _df_hdr.columns.tolist()
            _preview_cols_raw = [
                '#chr', 'pos(1-based)', 'ref', 'alt',
                'CADD_phred', 'SIFT_score', 'Polyphen2_HVAR_score',
                '1000Gp3_AF', 'REVEL_score', 'MetaSVM_score', 'PhyloP100way_vertebrate'
            ]
            _preview_cols_exist = [c for c in _preview_cols_raw if c in _all_cols]
            df_dbnsfp_preview = pd.read_csv(
                RAW_DBNSFP, sep='\t', compression='gzip',
                nrows=20, usecols=_preview_cols_exist, dtype=str
            )
            df_dbnsfp_preview.to_csv(dbnsfp_preview_path, index=False)
            print(f'  [OK] Preview dbNSFP ({len(df_dbnsfp_preview)} baris) tersimpan di:')
            print(f'       {dbnsfp_preview_path}')
        else:
            print(f'\n[LAMPIRAN] Preview dbNSFP sudah ada, dilewati: {dbnsfp_preview_path}')

    print('\nTahap 3 dilewati karena output sudah ada.')
    SKIP_PIPELINE = True
    print('\n[INFO] SKIP_PIPELINE aktif. Jalankan cell berikutnya, mereka akan otomatis di-skip.\n')

if not SKIP_PIPELINE:
    if not CLEAN_CLINVAR.exists():
        raise FileNotFoundError(f'File tidak ditemukan: {CLEAN_CLINVAR}. Pastikan Tahap 2 sudah selesai.')
    if not RAW_DBNSFP.exists():
        raise FileNotFoundError(f'File tidak ditemukan: {RAW_DBNSFP}. Pastikan Tahap 1 sudah selesai didownload.')


Checkpoint ditemukan: hasil gabungan sudah tersedia.
┌─────────────┬───────────────────────────────────────────────────────────────┐
│ Info        │ Nilai                                                         │
├─────────────┼───────────────────────────────────────────────────────────────┤
│ File        │ /content/drive/MyDrive/skripsi/data/clinvar_dbnsfp_merged.csv │
│ Total Baris │ 357,191                                                       │
│ Total Kolom │ 13                                                            │
└─────────────┴───────────────────────────────────────────────────────────────┘

20 Data Teratas (Cached):
┌─────────────┬─────┬─────────┬─────┬─────┬───────────────────┬───────┬──────────┬─────────┬───────┬───────┬────────┬────────────────────┐
│ VariationID │ chr │ pos     │ ref │ alt │ label             │ SIFT  │ PolyPhen │ MetaSVM │ REVEL │ CADD  │ PhyloP │ MAF                │
├─────────────┼─────┼─────────┼─────┼─────┼───────────────────┼───────┼──────────

 ## 2. MUAT DATA INPUT

In [ ]:
    # Membaca dataset clinvar yang sudah dibersihksn
    print('\nMemuat data ClinVar yang sudah dibersihkan...')
    df_clinvar = pd.read_csv(CLEAN_CLINVAR, dtype=str)
    df_clinvar['chr'] = df_clinvar['chr'].astype(str)
    df_clinvar['pos'] = df_clinvar['pos'].astype(str)
    print(f'Total varian ClinVar: {len(df_clinvar):,} baris x {df_clinvar.shape[1]} kolom')
    print('\n20 Data ClinVar (Input Sebelum Di-Join):')
    print_box_table(df_clinvar.head(20))


Memuat data ClinVar yang sudah dibersihkan...
Total varian ClinVar: 1,488,189 baris x 6 kolom

20 Data ClinVar (Input Sebelum Di-Join):
┌─────────────┬─────┬───────────┬─────┬─────┬───────────────────┐
│ VariationID │ chr │ pos       │ ref │ alt │ label             │
├─────────────┼─────┼───────────┼─────┼─────┼───────────────────┤
│ 5           │ 11  │ 126275389 │ C   │ T   │ Pathogenic        │
│ 6           │ 11  │ 126277517 │ A   │ G   │ Likely pathogenic │
│ 14          │ 6   │ 26093008  │ G   │ A   │ Benign            │
│ 18          │ 6   │ 26093215  │ G   │ T   │ Pathogenic        │
│ 19          │ 6   │ 26092916  │ A   │ C   │ Likely pathogenic │
│ 20          │ 2   │ 19989284  │ T   │ C   │ Pathogenic        │
│ 21          │ 2   │ 19945787  │ T   │ C   │ Pathogenic        │
│ 23          │ 2   │ 19933469  │ C   │ T   │ Likely pathogenic │
│ 27          │ 20  │ 25302322  │ G   │ A   │ Pathogenic        │
│ 30          │ 10  │ 97611535  │ G   │ T   │ Pathogenic        │
│ 31 

 ## 3. IDENTIFIKASI KOLOM dbNSFP


In [ ]:
    print('\nMengidentifikasi kolom dbNSFP...')
    # Mengintip daftar kolom tanpa memuat data
    df_header = pd.read_csv(RAW_DBNSFP, sep='\t', compression='gzip', nrows=0)
    dbnsfp_columns = df_header.columns.tolist()

    # Membuat lampiran preview
    print('\n[LAMPIRAN] Membaca preview isi raw dbNSFP (20 baris, kolom relevan)...')
    _preview_cols_raw = [
        '#chr', 'pos(1-based)', 'ref', 'alt',
        'CADD_phred', 'SIFT_score', 'Polyphen2_HVAR_score',
        '1000Gp3_AF', 'REVEL_score', 'MetaSVM_score', 'PhyloP100way_vertebrate'
    ]
    _preview_cols_exist = [c for c in _preview_cols_raw if c in dbnsfp_columns]
    df_dbnsfp_preview = pd.read_csv(
        RAW_DBNSFP, sep='\t', compression='gzip',
        nrows=20, usecols=_preview_cols_exist, dtype=str
    )
    dbnsfp_preview_path = LAMPIRAN_DIR / 'dbnsfp_raw_preview_top20.csv'
    df_dbnsfp_preview.to_csv(dbnsfp_preview_path, index=False)
    print(f'  [OK] Preview dbNSFP ({len(df_dbnsfp_preview)} baris, {df_dbnsfp_preview.shape[1]} kolom) tersimpan di:')
    print(f'       {dbnsfp_preview_path}')
    print_box_table(df_dbnsfp_preview.head(5))
    del df_dbnsfp_preview

    # Mencari kolom otomatis dan fleksibel
    def find_col(keywords: list[str]) -> str | None:
        for col in dbnsfp_columns:
            if any(kw.lower() in col.lower() for kw in keywords):
                return col
        return None

    col_chr      = '#chr'             if '#chr'              in dbnsfp_columns else find_col(['hg38_chr', 'chr'])
    col_pos      = 'pos(1-based)'     if 'pos(1-based)'      in dbnsfp_columns else find_col(['hg38_pos', 'pos'])
    col_ref      = 'ref'
    col_alt      = 'alt'
    col_cadd     = find_col(['CADD_phred'])
    col_sift     = find_col(['SIFT_score'])
    col_polyphen = find_col(['Polyphen2_HVAR_score', 'Polyphen2_HDIV_score'])
    col_maf      = find_col(['1000Gp3_AF'])
    col_revel    = find_col(['REVEL_score'])
    col_metasvm  = find_col(['MetaSVM_score'])
    col_phylop   = find_col(['PhyloP100way_vertebrate', 'phyloP100way_vertebrate'])

    FALLBACK_NAMES = {
        'col_chr'      : '#chr',
        'col_pos'      : 'pos(1-based)',
        'col_ref'      : 'ref',
        'col_alt'      : 'alt',
        'col_cadd'     : 'CADD_phred',
        'col_sift'     : 'SIFT_score',
        'col_polyphen' : 'Polyphen2_HVAR_score',
        'col_maf'      : '1000Gp3_AF',
        'col_revel'    : 'REVEL_score',
        'col_metasvm'  : 'MetaSVM_score',
        'col_phylop'   : 'PhyloP100way_vertebrate',
    }
    col_vars = {
        'col_chr': col_chr, 'col_pos': col_pos, 'col_ref': col_ref, 'col_alt': col_alt,
        'col_cadd': col_cadd, 'col_sift': col_sift, 'col_polyphen': col_polyphen,
        'col_maf': col_maf, 'col_revel': col_revel, 'col_metasvm': col_metasvm,
        'col_phylop': col_phylop,
    }
    for var_name, val in col_vars.items():
        if val is None:
            fallback = FALLBACK_NAMES[var_name]
            print(f'  [PERINGATAN] Kolom untuk {var_name} tidak ditemukan, fallback ke: {fallback}')
            col_vars[var_name] = fallback

    col_chr, col_pos, col_ref, col_alt = col_vars['col_chr'], col_vars['col_pos'], col_vars['col_ref'], col_vars['col_alt']
    col_cadd, col_sift, col_polyphen, col_maf = col_vars['col_cadd'], col_vars['col_sift'], col_vars['col_polyphen'], col_vars['col_maf']
    col_revel, col_metasvm, col_phylop = col_vars['col_revel'], col_vars['col_metasvm'], col_vars['col_phylop']

    cols_to_use = [col_chr, col_pos, col_ref, col_alt, col_cadd, col_sift, col_polyphen, col_maf, col_revel, col_metasvm, col_phylop]
    cols_to_use = list(dict.fromkeys(cols_to_use))

    print('\nKolom yang akan diambil dari dbNSFP:')
    print_box_table(pd.DataFrame({
        'Nama Kolom dbNSFP' : cols_to_use,
        'Alias di Pipeline' : ['chr', 'pos', 'ref', 'alt', 'CADD', 'SIFT', 'PolyPhen', 'MAF', 'REVEL', 'MetaSVM', 'PhyloP'][:len(cols_to_use)],
    }))

    RENAME_MAP = {
        col_chr      : 'chr',
        col_pos      : 'pos',
        col_cadd     : 'CADD',
        col_sift     : 'SIFT',
        col_polyphen : 'PolyPhen',
        col_maf      : 'MAF',
        col_revel    : 'REVEL',
        col_metasvm  : 'MetaSVM',
        col_phylop   : 'PhyloP',
    }

    RENAME_MAP = {k: v for k, v in RENAME_MAP.items() if k != v}

    cols_to_use_raw = [
        col_chr, col_pos, col_ref, col_alt,
        col_cadd, col_sift, col_polyphen, col_maf,
        col_revel, col_metasvm, col_phylop,
    ]
    cols_to_use = list(dict.fromkeys(c for c in cols_to_use_raw if c is not None and c in dbnsfp_columns))

    FEATURE_COLS = ['CADD', 'SIFT', 'PolyPhen', 'MAF', 'REVEL', 'MetaSVM', 'PhyloP']



Mengidentifikasi kolom dbNSFP...

[LAMPIRAN] Membaca preview isi raw dbNSFP (20 baris, kolom relevan)...
  [OK] Preview dbNSFP (20 baris, 10 kolom) tersimpan di:
       /content/drive/MyDrive/skripsi/data/Lampiran_Skripsi/dbnsfp_raw_preview_top20.csv
┌──────┬──────────────┬─────┬─────┬────────────┬──────────────────────┬───────────────┬─────────────┬────────────┬────────────┐
│ #chr │ pos(1-based) │ ref │ alt │ SIFT_score │ Polyphen2_HVAR_score │ MetaSVM_score │ REVEL_score │ CADD_phred │ 1000Gp3_AF │
├──────┼──────────────┼─────┼─────┼────────────┼──────────────────────┼───────────────┼─────────────┼────────────┼────────────┤
│ 1    │ 65565        │ A   │ C   │ .          │ .                    │ .             │ .           │ 17.18      │ .          │
│ 1    │ 65565        │ A   │ G   │ .          │ .                    │ .             │ .           │ 19.49      │ .          │
│ 1    │ 65565        │ A   │ T   │ .          │ .                    │ .             │ .           │ 17.37 

## 4. PROSES JOIN PER CHUNK + PARSING MULTI-NILAI
    

In [ ]:
    print(f'\nMulai memproses dbNSFP. Proses ini panjang, checkpoint aktif setiap 10 chunk.')

    # Mengatur agar membaca per 500.000 baris
    chunk_size = 500_000
    save_every = 10
    start_chunk_idx = 0
    matched_chunks: list[pd.DataFrame] = []

    # Lanjutkan otomatis ketika sebelumnya mati
    if CHECKPOINT_MERGED.exists() and CHECKPOINT_META.exists():
        with open(CHECKPOINT_META, 'r', encoding='utf-8') as f:
            start_chunk_idx = int(f.read().strip())
        print(f'Resume terdeteksi dari chunk ke-{start_chunk_idx}.')
        matched_chunks.append(pd.read_csv(CHECKPOINT_MERGED, dtype=str))
    if not SKIP_PIPELINE:
        print('Memulai anotasi dari awal (tidak ada checkpoint).')

    df_clinvar['pos'] = df_clinvar['pos'].astype(str)
    start_time = time.time()
    total_rows_processed = 0

    reader = pd.read_csv(
        RAW_DBNSFP,
        sep='\t',
        compression='gzip',
        usecols=cols_to_use,
        dtype=str,
        chunksize=chunk_size,
    )

    for chunk_index, chunk in enumerate(reader):
        if chunk_index < start_chunk_idx:
            continue

        total_rows_processed += len(chunk)
        chunk = chunk.rename(columns=RENAME_MAP)
        chunk['chr'] = chunk['chr'].astype(str)
        chunk['pos'] = chunk['pos'].astype(str)

        for feat in FEATURE_COLS:
            if feat in chunk.columns:
                chunk[feat] = chunk[feat].apply(extract_max_score)

        matched = pd.merge(df_clinvar, chunk, on=['chr', 'pos', 'ref', 'alt'], how='inner')
        if not matched.empty:
            matched_chunks.append(matched)

        if (chunk_index + 1) % save_every == 0 and matched_chunks:
            checkpoint_df = pd.concat(matched_chunks, ignore_index=True).drop_duplicates()
            checkpoint_df.to_csv(CHECKPOINT_MERGED, index=False)
            with open(CHECKPOINT_META, 'w', encoding='utf-8') as f:
                f.write(str(chunk_index + 1))
            elapsed = (time.time() - start_time) / 60
            print(f'Checkpoint tersimpan di chunk {chunk_index + 1} | '
                  f'match: {len(checkpoint_df):,} | waktu: {elapsed:.1f} menit')

        del chunk
        gc.collect()


Mulai memproses dbNSFP. Proses ini panjang, checkpoint aktif setiap 10 chunk.
Resume terdeteksi dari chunk ke-170.


## 5. SIMPAN HASIL AKHIR

In [ ]:
    if not matched_chunks:
        raise ValueError('Tidak ada mutasi yang cocok dengan dbNSFP!')

    final_df = pd.concat(matched_chunks, ignore_index=True).drop_duplicates()

    cols_to_drop = [c for c in [col_chr, col_pos, col_ref, col_alt]
                    if c is not None and c in final_df.columns and c not in ['chr', 'pos', 'ref', 'alt']]
    final_df = final_df.drop(columns=cols_to_drop, errors='ignore')

    for feat in FEATURE_COLS:
        if feat in final_df.columns:
            final_df[feat] = pd.to_numeric(final_df[feat], errors='coerce')
    if not SKIP_PIPELINE:
            print(f'  [INFO] Fitur "{feat}" tidak ada di dbNSFP ini — dilewati (tidak masuk CSV).')

    final_df.to_csv(FINAL_MERGED, index=False)

    if CHECKPOINT_MERGED.exists():
        CHECKPOINT_MERGED.unlink()
    if CHECKPOINT_META.exists():
        CHECKPOINT_META.unlink()

    elapsed_total = (time.time() - start_time) / 60
    print(f'\n[SELESAI] Data gabungan berhasil disimpan di:\n   {FINAL_MERGED}')
    print_kv_table({
        'Total Data Final'    : f'{len(final_df):,} baris',
        'Total Kolom'         : str(final_df.shape[1]),
        'Fitur Tersedia'      : ', '.join([f for f in FEATURE_COLS if f in final_df.columns]),
        'Waktu Total'         : f'{elapsed_total:.1f} menit',
    }, key_label='Info', val_label='Nilai')

    print('\n20 Data Teratas (Final):')
    print_box_table(final_df.head(20))

    top20_path = LAMPIRAN_DIR / 'clinvar_dbnsfp_merged_top20.csv'
    final_df.head(20).to_csv(top20_path, index=False)
    print(f'Top 20 data (untuk dosen) tersimpan di:\n   {top20_path}')

    print('\nDistribusi Kelas Label:')
    dist = final_df['label'].value_counts()
    print_kv_table({k: f'{v:,}' for k, v in dist.items()}, key_label='Label', val_label='Jumlah')


[SELESAI] Data gabungan berhasil disimpan di:
   /content/drive/MyDrive/skripsi/data/clinvar_dbnsfp_merged.csv
┌──────────────────┬───────────────────────────────────────────────────┐
│ Info             │ Nilai                                             │
├──────────────────┼───────────────────────────────────────────────────┤
│ Total Data Final │ 357,191 baris                                     │
│ Total Kolom      │ 13                                                │
│ Fitur Tersedia   │ CADD, SIFT, PolyPhen, MAF, REVEL, MetaSVM, PhyloP │
│ Waktu Total      │ 77.2 menit                                        │
└──────────────────┴───────────────────────────────────────────────────┘

20 Data Teratas (Final):
┌─────────────┬─────┬─────────┬─────┬─────┬───────────────────┬───────┬──────────┬─────────┬───────┬───────┬────────┬────────────────────┐
│ VariationID │ chr │ pos     │ ref │ alt │ label             │ SIFT  │ PolyPhen │ MetaSVM │ REVEL │ CADD  │ PhyloP │ MAF                │


## 6. BERSIHKAN MEMORI

In [ ]:
    del matched_chunks, df_clinvar
    if 'final_df' in locals():
        del final_df
    gc.collect()
    print('\nTAHAP 3 SELESAI!')


TAHAP 3 SELESAI!
